# Connectivity 2 — Graph neural network on GLM connectivity graphs

| | |
|---|---|
| **Original** | `Liam_work/tests/graphical_analysis/AD_GNN_Classification.ipynb` |
| **Status** | Side project (Feb 2026). Animal-level classifier (only 20 graphs). |

Each animal's within-region GLM graphs (nodes = neurons with cell-type / rate / degree features, edges =
coupling weights) → small GCN (PyTorch Geometric) → AD vs WT. Leave-one-animal-out, ALL / OLD / YOUNG with
the older **212-day** cutoff. Compared with a hand-crafted-graph-features + logistic-regression baseline,
and checked for seed stability.

Needs `torch` + `torch_geometric`. Reads `data/glm_results/glm_connectivity_results_20animals_REM.pkl`.

Paths: raw inputs are read from `/media/HlabShare/...` (read-only). Anything this copy writes goes under `/hlabhome/wg-mjames/git/LDCODES12_ind_neuron_spiking_classification_project/` — never back into `Liam_work/`.

---
*Repo copy: outputs stripped, header/section notes added, lab preamble added where missing, and output
paths moved into this repo. Analysis logic is unchanged from Liam's version.*

In [ ]:
# =============================================================================
# LAB-WIDE PREAMBLE — CPU thread limits + figure-export font config
# (added in the repo copy; set BEFORE numpy/scipy/sklearn/xgboost are imported)
# =============================================================================
import os
import matplotlib.pyplot as plt

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams.update({'font.family': 'DejaVu Sans'})

# Limit CPU threads — shared server "marvin"
os.environ['OMP_NUM_THREADS']        = '4'
os.environ['OPENBLAS_NUM_THREADS']   = '4'
os.environ['MKL_NUM_THREADS']        = '4'
os.environ['VECLIB_MAXIMUM_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS']    = '4'

# AD vs WT Classification: Graph Neural Network
## Using Point Process GLM Connectivity Graphs

### Approach
Each animal has **within-region connectivity graphs** from Point Process GLM (estimated during REM sleep).  
Nodes = neurons, edges = directed functional connections (GLM coupling weights).  

We build a **Graph Neural Network** that:
1. Takes each animal's regional graphs as input
2. Uses node features (cell type, firing rate, connection degree) and edge weights
3. Pools node embeddings into a graph-level representation
4. Classifies AD vs WT

### Three splits (Leave-One-Animal-Out CV):
- **ALL**: 20 animals (11 WT, 9 AD)
- **OLD** (>212d): 6 WT + 4 AD
- **YOUNG** (<=212d): 5 WT + 5 AD

### Data
- `glm_connectivity_results_20animals_REM.pkl`: Pre-computed GLM connectivity
- 4 brain regions: AC, CA1, RSC/RSPv, V1
- Directed, weighted edges (excitatory & inhibitory)

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import time
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool, global_max_pool
from torch_geometric.utils import to_undirected

from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, confusion_matrix
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-whitegrid')
np.random.seed(42)
torch.manual_seed(42)

AGE_CUTOFF = 212
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('PyTorch:', torch.__version__)

# Animal metadata (ages from ISI dataset)
ANIMAL_INFO = {
    'BTH100': {'geno': 'wt', 'age': 98.2},
    'BTH90':  {'geno': 'wt', 'age': 101.2},
    'BTH95':  {'geno': 'wt', 'age': 105.1},
    'CAF104': {'geno': 'wt', 'age': 387.2},
    'CAF42':  {'geno': 'wt', 'age': 207.4},
    'CAF69':  {'geno': 'wt', 'age': 358.2},
    'CAF71':  {'geno': 'app_ps1', 'age': 492.3},
    'CAF73':  {'geno': 'app_ps1', 'age': 381.3},
    'CAF74':  {'geno': 'app_ps1', 'age': 419.4},
    'CAF75':  {'geno': 'app_ps1', 'age': 388.5},
    'CAF77':  {'geno': 'wt', 'age': 394.3},
    'CAF81':  {'geno': 'wt', 'age': 455.2},
    'CAF82':  {'geno': 'wt', 'age': 455.2},
    'CAF88':  {'geno': 'wt', 'age': 532.4},
    'ZBR106': {'geno': 'wt', 'age': 98.6},
    'ZBR113': {'geno': 'app_ps1', 'age': 90.5},
    'ZBR114': {'geno': 'app_ps1', 'age': 73.4},
    'ZBR117': {'geno': 'app_ps1', 'age': 79.4},
    'ZBR118': {'geno': 'app_ps1', 'age': 99.3},
    'ZBR119': {'geno': 'app_ps1', 'age': 112.2},
}

# Region name normalization (ACC->AC, ACad->AC, RSC->RSC, RSPv->RSC)
REGION_MAP = {'ACC': 'AC', 'ACad': 'AC', 'RSC': 'RSC', 'RSPv': 'RSC', 'CA1': 'CA1', 'V1': 'V1'}
REGIONS = ['AC', 'CA1', 'RSC', 'V1']

In [ ]:
# =============================================================================
# LOAD GLM CONNECTIVITY AND BUILD PyG GRAPHS
# =============================================================================
print('Loading GLM connectivity results...')
with open('/hlabhome/wg-mjames/git/LDCODES12_ind_neuron_spiking_classification_project/data/glm_results/glm_connectivity_results_20animals_REM.pkl', 'rb') as f:
    glm_data = pickle.load(f)


def build_animal_graph(animal_name, animal_data):
    """
    Build a single PyG Data object for one animal.
    
    Combines all regional subgraphs into one graph with:
    - Node features: [cell_type_RSU, cell_type_FS, log_firing_rate,
                      in_degree, out_degree, region_onehot x4]
    - Edge features: [weight, abs_weight, is_excitatory]
    """
    all_node_feats = []
    all_edge_index = []  # (2, E)
    all_edge_attr = []
    node_offset = 0
    
    neuron_info = animal_data['neuron_info']
    
    for orig_region, rdata in animal_data['regions'].items():
        region = REGION_MAP[orig_region]
        region_idx = REGIONS.index(region)
        conn = rdata['connectivity']  # (N, N)
        indices = rdata['neuron_indices']
        N = conn.shape[0]
        
        # Node features for this region
        for local_i, global_i in enumerate(indices):
            info = neuron_info[global_i]
            is_rsu = 1.0 if info['cell_type'] == 'RSU' else 0.0
            is_fs = 1.0 if info['cell_type'] == 'FS' else 0.0
            fr = np.log1p(info['firing_rate_hz'])
            
            # Degree features from connectivity
            out_deg = np.sum(np.abs(conn[local_i, :]) > 1e-6)
            in_deg = np.sum(np.abs(conn[:, local_i]) > 1e-6)
            out_strength = np.sum(np.abs(conn[local_i, :]))
            in_strength = np.sum(np.abs(conn[:, local_i]))
            
            # Region one-hot
            region_oh = [0.0] * 4
            region_oh[region_idx] = 1.0
            
            feat = [is_rsu, is_fs, fr, 
                    out_deg / max(N - 1, 1), in_deg / max(N - 1, 1),
                    out_strength, in_strength] + region_oh
            all_node_feats.append(feat)
        
        # Edges for this region
        for i in range(N):
            for j in range(N):
                if i == j:
                    continue
                w = conn[i, j]
                if abs(w) > 1e-6:
                    all_edge_index.append([node_offset + i, node_offset + j])
                    all_edge_attr.append([w, abs(w), 1.0 if w > 0 else 0.0])
        
        node_offset += N
    
    # Convert to tensors
    x = torch.tensor(all_node_feats, dtype=torch.float)
    
    if len(all_edge_index) > 0:
        edge_index = torch.tensor(all_edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(all_edge_attr, dtype=torch.float)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 3), dtype=torch.float)
    
    # Label
    info = ANIMAL_INFO[animal_name]
    label = 1 if info['geno'] == 'app_ps1' else 0
    age_group = 'OLD' if info['age'] > AGE_CUTOFF else 'YOUNG'
    
    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=torch.tensor([label], dtype=torch.long),
    )
    data.animal = animal_name
    data.genotype = info['geno']
    data.age = info['age']
    data.age_group = age_group
    data.n_neurons = x.shape[0]
    
    return data


# Build all graphs
graphs = []
for aname in sorted(glm_data.keys()):
    g = build_animal_graph(aname, glm_data[aname])
    graphs.append(g)
    n_exc = (g.edge_attr[:, 2] > 0.5).sum().item() if g.edge_attr.shape[0] > 0 else 0
    n_inh = g.edge_attr.shape[0] - n_exc
    print('{}: {} nodes, {} edges ({}E/{}I), label={} ({})'.format(
        aname, g.n_neurons, g.edge_index.shape[1],
        n_exc, n_inh, g.y.item(),
        'AD' if g.y.item() == 1 else 'WT'))

print('\nBuilt {} animal graphs'.format(len(graphs)))
print('Node feature dim: {}'.format(graphs[0].x.shape[1]))
print('Edge feature dim: {}'.format(graphs[0].edge_attr.shape[1]))

In [ ]:
# =============================================================================
# GRAPH STATISTICS VISUALIZATION
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Node counts by genotype
ax = axes[0]
wt_nodes = [g.n_neurons for g in graphs if g.y.item() == 0]
ad_nodes = [g.n_neurons for g in graphs if g.y.item() == 1]
ax.boxplot([wt_nodes, ad_nodes], labels=['WT', 'AD'])
ax.set_ylabel('Number of neurons')
ax.set_title('Neurons per Animal')
for i, (nodes, label) in enumerate([(wt_nodes, 'WT'), (ad_nodes, 'AD')]):
    ax.scatter([i+1]*len(nodes), nodes, alpha=0.5, zorder=5)

# 2. Edge counts
ax = axes[1]
wt_edges = [g.edge_index.shape[1] for g in graphs if g.y.item() == 0]
ad_edges = [g.edge_index.shape[1] for g in graphs if g.y.item() == 1]
ax.boxplot([wt_edges, ad_edges], labels=['WT', 'AD'])
ax.set_ylabel('Number of edges')
ax.set_title('Connections per Animal')
for i, edges in enumerate([wt_edges, ad_edges]):
    ax.scatter([i+1]*len(edges), edges, alpha=0.5, zorder=5)

# 3. Connection density
ax = axes[2]
wt_dens = [g.edge_index.shape[1] / max(g.n_neurons * (g.n_neurons - 1), 1) for g in graphs if g.y.item() == 0]
ad_dens = [g.edge_index.shape[1] / max(g.n_neurons * (g.n_neurons - 1), 1) for g in graphs if g.y.item() == 1]
ax.boxplot([wt_dens, ad_dens], labels=['WT', 'AD'])
ax.set_ylabel('Density')
ax.set_title('Graph Density (within-region)')
for i, dens in enumerate([wt_dens, ad_dens]):
    ax.scatter([i+1]*len(dens), dens, alpha=0.5, zorder=5)

plt.suptitle('Graph Structure: WT vs AD', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## GNN Architecture

**GCN-based graph classifier:**
1. 3 GCN layers with residual connections and batch norm
2. Mean + max pooling to get graph-level embedding
3. 2-layer MLP classifier head

Since we only have 20 graphs, we keep the model small to avoid overfitting.

In [ ]:
# =============================================================================
# GNN MODEL
# =============================================================================

class GNNClassifier(nn.Module):
    """Graph-level classifier using GCN layers + readout pooling."""
    
    def __init__(self, in_channels, hidden_channels=64, num_layers=3, dropout=0.3):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        # First conv
        self.convs.append(GCNConv(in_channels, hidden_channels))
        self.bns.append(nn.BatchNorm1d(hidden_channels))
        
        # Hidden convs
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
            self.bns.append(nn.BatchNorm1d(hidden_channels))
        
        # Classifier head (mean + max pool = 2 * hidden)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, 2)
        )
        self.dropout = dropout
    
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        for i, (conv, bn) in enumerate(zip(self.convs, self.bns)):
            x_new = conv(x, edge_index)
            x_new = bn(x_new)
            x_new = F.relu(x_new)
            x_new = F.dropout(x_new, p=self.dropout, training=self.training)
            # Residual connection after first layer
            if i > 0:
                x = x + x_new
            else:
                x = x_new
        
        # Graph-level readout: mean + max
        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        x_graph = torch.cat([x_mean, x_max], dim=1)
        
        return self.classifier(x_graph)
    
    def get_embedding(self, data):
        """Get graph-level embedding (before classifier)."""
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for i, (conv, bn) in enumerate(zip(self.convs, self.bns)):
            x_new = conv(x, edge_index)
            x_new = bn(x_new)
            x_new = F.relu(x_new)
            if i > 0:
                x = x + x_new
            else:
                x = x_new
        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        return torch.cat([x_mean, x_max], dim=1)


print('GNN model defined.')
print('Architecture: {} GCN layers, {} hidden dim, mean+max pool'.format(3, 64))

In [ ]:
# =============================================================================
# TRAINING AND EVALUATION UTILITIES
# =============================================================================

def train_epoch(model, train_loader, optimizer, criterion):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        out = model(batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(train_loader.dataset)


def evaluate(model, data_list):
    """Evaluate on a list of Data objects. Returns pred probs and labels."""
    model.eval()
    probs = []
    labels = []
    with torch.no_grad():
        for data in data_list:
            data_dev = data.clone().to(DEVICE)
            # Need batch vector for single graph
            if not hasattr(data_dev, 'batch') or data_dev.batch is None:
                data_dev.batch = torch.zeros(data_dev.x.shape[0], dtype=torch.long, device=DEVICE)
            out = model(data_dev)
            prob = F.softmax(out, dim=1)[0, 1].item()
            probs.append(prob)
            labels.append(data.y.item())
    return np.array(probs), np.array(labels)


def run_loao_cv(graph_list, n_epochs=200, hidden_channels=64, num_layers=3,
                lr=0.001, weight_decay=1e-3, dropout=0.3, n_runs=5, title='GNN'):
    """
    Leave-One-Animal-Out CV with multiple random seeds.
    
    Since we have only 20 graphs, we run multiple times with different
    seeds and average to reduce variance from initialization.
    """
    n_animals = len(graph_list)
    in_channels = graph_list[0].x.shape[1]
    
    all_run_probs = np.zeros((n_runs, n_animals))
    labels = np.array([g.y.item() for g in graph_list])
    
    print('  {} animals, {} features, {} runs x {} folds'.format(
        n_animals, in_channels, n_runs, n_animals))
    
    t0 = time.time()
    
    for run in range(n_runs):
        seed = 42 + run
        torch.manual_seed(seed)
        np.random.seed(seed)
        
        for fold_idx in range(n_animals):
            # Leave one out
            test_graph = graph_list[fold_idx]
            train_graphs = [g for i, g in enumerate(graph_list) if i != fold_idx]
            
            # Build model
            model = GNNClassifier(
                in_channels=in_channels,
                hidden_channels=hidden_channels,
                num_layers=num_layers,
                dropout=dropout
            ).to(DEVICE)
            
            # Class weights for imbalanced data
            train_labels = [g.y.item() for g in train_graphs]
            n_pos = sum(train_labels)
            n_neg = len(train_labels) - n_pos
            weight = torch.tensor([1.0 / max(n_neg, 1), 1.0 / max(n_pos, 1)],
                                  dtype=torch.float, device=DEVICE)
            weight = weight / weight.sum() * 2
            criterion = nn.CrossEntropyLoss(weight=weight)
            
            optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
            
            # Create batch from training graphs (clone to avoid moving originals to GPU)
            train_graphs_cpu = [g.clone() for g in train_graphs]
            train_batch = Batch.from_data_list(train_graphs_cpu)
            
            # Train
            best_loss = float('inf')
            patience_counter = 0
            for epoch in range(n_epochs):
                model.train()
                train_batch_dev = train_batch.to(DEVICE)
                optimizer.zero_grad()
                out = model(train_batch_dev)
                loss = criterion(out, train_batch_dev.y)
                loss.backward()
                optimizer.step()
                scheduler.step()
                
                if loss.item() < best_loss - 1e-4:
                    best_loss = loss.item()
                    patience_counter = 0
                else:
                    patience_counter += 1
                if patience_counter >= 30:
                    break
            
            # Evaluate on held-out animal
            prob, _ = evaluate(model, [test_graph])
            all_run_probs[run, fold_idx] = prob[0]
        
        run_preds = (all_run_probs[run] > 0.5).astype(int)
        run_acc = accuracy_score(labels, run_preds)
        run_auc = roc_auc_score(labels, all_run_probs[run])
        print('  Run {}/{}: Acc={:.1%} AUC={:.3f}'.format(run + 1, n_runs, run_acc, run_auc))
    
    elapsed = time.time() - t0
    
    # Average across runs
    mean_probs = all_run_probs.mean(axis=0)
    mean_preds = (mean_probs > 0.5).astype(int)
    
    acc = accuracy_score(labels, mean_preds)
    auc = roc_auc_score(labels, mean_probs)
    
    print('\n  --- {} (averaged over {} runs) ---'.format(title, n_runs))
    print('  Accuracy: {:.1%} ({}/{})  AUC: {:.3f}'.format(
        acc, sum(labels == mean_preds), len(labels), auc))
    print('  Time: {:.1f}s'.format(elapsed))
    
    # Per-animal details
    animals = [g.animal for g in graph_list]
    genos = ['AD' if g.y.item() == 1 else 'WT' for g in graph_list]
    for i in range(n_animals):
        mark = 'Y' if labels[i] == mean_preds[i] else 'X'
        print('    {} ({}) prob={:.3f} [{}]'.format(animals[i], genos[i], mean_probs[i], mark))
    
    return {
        'acc': acc, 'auc': auc,
        'labels': labels, 'probs': mean_probs, 'preds': mean_preds,
        'animals': animals, 'genos': genos,
        'all_run_probs': all_run_probs,
        'title': title
    }


def run_all_three_splits(graph_list, title, **kwargs):
    """Run LOAO CV on ALL, OLD, YOUNG splits."""
    results = {}
    
    print('\n' + '=' * 70)
    print('{} -- ALL ANIMALS'.format(title))
    print('=' * 70)
    results['all'] = run_loao_cv(graph_list, title='{} (All)'.format(title), **kwargs)
    
    for age_grp in ['OLD', 'YOUNG']:
        print('\n' + '=' * 70)
        print('{} -- {} ONLY'.format(title, age_grp))
        print('=' * 70)
        subset = [g for g in graph_list if g.age_group == age_grp]
        n_ad = sum(1 for g in subset if g.y.item() == 1)
        n_wt = len(subset) - n_ad
        print('  {} animals: {} WT, {} AD'.format(len(subset), n_wt, n_ad))
        if n_ad < 2 or n_wt < 2:
            print('  SKIPPED: need >= 2 per class')
            continue
        results[age_grp.lower()] = run_loao_cv(
            subset, title='{} ({})'.format(title, age_grp), **kwargs)
    
    return results


print('Training utilities defined.')

---
## Run GNN Classification

In [ ]:
# =============================================================================
# RUN GNN — ALL, OLD, YOUNG
# =============================================================================
gnn_results = run_all_three_splits(
    graphs, 'GNN (GCN)',
    n_epochs=200, hidden_channels=64, num_layers=3,
    lr=0.001, weight_decay=1e-3, dropout=0.3, n_runs=5
)

In [ ]:
# =============================================================================
# VISUALIZATION
# =============================================================================

def plot_gnn_results(results_dict, suptitle):
    """Plot ROC, confusion matrix, and per-animal probs."""
    splits = [k for k in ['all', 'old', 'young'] if k in results_dict]
    n_splits = len(splits)
    fig, axes = plt.subplots(n_splits, 3, figsize=(16, 5 * n_splits))
    if n_splits == 1:
        axes = axes.reshape(1, -1)
    
    split_labels = {'all': 'All Animals', 'old': 'OLD (>212d)', 'young': 'YOUNG (<=212d)'}
    
    for row, split in enumerate(splits):
        res = results_dict[split]
        
        # ROC
        ax = axes[row, 0]
        fpr, tpr, _ = roc_curve(res['labels'], res['probs'])
        ax.plot(fpr, tpr, 'b-', lw=2, label='AUC={:.3f}'.format(res['auc']))
        ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
        ax.set_xlabel('FPR')
        ax.set_ylabel('TPR')
        ax.set_title('{}: ROC'.format(split_labels[split]), fontweight='bold')
        ax.legend(fontsize=11)
        
        # Confusion matrix
        ax = axes[row, 1]
        cm = confusion_matrix(res['labels'], res['preds'])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['WT', 'AD'], yticklabels=['WT', 'AD'])
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.set_title('{}: Confusion'.format(split_labels[split]))
        
        # Per-animal bar
        ax = axes[row, 2]
        order = np.argsort(res['probs'])
        colors = ['crimson' if res['genos'][i] == 'AD' else 'steelblue' for i in order]
        ax.barh(range(len(order)), res['probs'][order], color=colors, edgecolor='black')
        ax.set_yticks(range(len(order)))
        ax.set_yticklabels(['{} ({})'.format(res['animals'][i], res['genos'][i]) for i in order])
        ax.axvline(x=0.5, color='black', linestyle='--', lw=1.5)
        ax.set_xlabel('P(AD)')
        ax.set_title('{}: Per-Animal Prob'.format(split_labels[split]))
    
    fig.suptitle(suptitle, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


plot_gnn_results(gnn_results, 'GNN (GCN) Classification: AD vs WT')

In [ ]:
# =============================================================================
# RUN VARIANCE ANALYSIS — how stable are predictions across seeds?
# =============================================================================
print('=' * 70)
print('PREDICTION STABILITY ACROSS RUNS')
print('=' * 70)

if 'all' in gnn_results:
    res = gnn_results['all']
    run_probs = res['all_run_probs']  # (n_runs, n_animals)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    animals = res['animals']
    genos = res['genos']
    mean_p = run_probs.mean(axis=0)
    std_p = run_probs.std(axis=0)
    order = np.argsort(mean_p)
    
    colors = ['crimson' if genos[i] == 'AD' else 'steelblue' for i in order]
    ax.barh(range(len(order)), mean_p[order], xerr=std_p[order],
            color=colors, edgecolor='black', capsize=3)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(['{} ({})'.format(animals[i], genos[i]) for i in order])
    ax.axvline(x=0.5, color='black', linestyle='--', lw=1.5)
    ax.set_xlabel('P(AD) mean +/- std across {} runs'.format(run_probs.shape[0]))
    ax.set_title('Prediction Stability (error bars = across-run std)', fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print('\nPer-animal stability:')
    for i in order:
        print('  {} ({}): {:.3f} +/- {:.3f}  [range: {:.3f}-{:.3f}]'.format(
            animals[i], genos[i], mean_p[i], std_p[i],
            run_probs[:, i].min(), run_probs[:, i].max()))

---
## Baseline Comparison: Graph Features + Logistic Regression

Compare GNN to a simpler baseline that extracts hand-crafted graph features
and feeds them to logistic regression.

In [ ]:
# =============================================================================
# BASELINE: HAND-CRAFTED GRAPH FEATURES + LOGISTIC REGRESSION
# =============================================================================
from sklearn.linear_model import LogisticRegression


def extract_graph_features(graph_list):
    """Extract hand-crafted features from each graph."""
    records = []
    for g in graph_list:
        feat = {}
        n_edges = g.edge_index.shape[1]
        feat['density'] = n_edges / max(g.n_neurons * (g.n_neurons - 1), 1)
        
        if n_edges > 0:
            weights = g.edge_attr[:, 0].numpy()
            abs_weights = g.edge_attr[:, 1].numpy()
            exc_mask = g.edge_attr[:, 2].numpy() > 0.5
            
            feat['mean_abs_weight'] = np.mean(abs_weights)
            feat['std_abs_weight'] = np.std(abs_weights)
            feat['mean_exc_weight'] = np.mean(weights[exc_mask]) if exc_mask.sum() > 0 else 0
            feat['mean_inh_weight'] = np.mean(weights[~exc_mask]) if (~exc_mask).sum() > 0 else 0
            feat['exc_frac'] = exc_mask.mean()
            feat['ei_ratio'] = exc_mask.sum() / max((~exc_mask).sum(), 1)
            feat['weight_skew'] = stats.skew(weights)
            feat['weight_kurt'] = stats.kurtosis(weights)
        else:
            for k in ['mean_abs_weight', 'std_abs_weight', 'mean_exc_weight',
                      'mean_inh_weight', 'exc_frac', 'ei_ratio', 'weight_skew', 'weight_kurt']:
                feat[k] = 0
        
        # Node feature stats (only normalized/biological ones)
        x = g.x.numpy()
        feat['mean_out_degree'] = x[:, 3].mean()
        feat['mean_in_degree'] = x[:, 4].mean()
        
        feat['label'] = g.y.item()
        feat['animal'] = g.animal
        feat['age_group'] = g.age_group
        records.append(feat)
    
    return pd.DataFrame(records)


def run_logreg_loao(feat_df, feat_cols, title='LogReg'):
    """LOAO with logistic regression."""
    animals = feat_df['animal'].values
    X = feat_df[feat_cols].values
    y = feat_df['label'].values
    
    # Fill NaNs
    for col in range(X.shape[1]):
        mask = np.isnan(X[:, col])
        if mask.any():
            X[mask, col] = np.nanmedian(X[:, col])
    
    probs = np.zeros(len(y))
    for i in range(len(y)):
        train_mask = np.arange(len(y)) != i
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X[train_mask])
        X_test = scaler.transform(X[i:i+1])
        lr = LogisticRegression(C=0.1, penalty='l2', solver='lbfgs', max_iter=1000, random_state=42)
        lr.fit(X_train, y[train_mask])
        probs[i] = lr.predict_proba(X_test)[0, 1]
    
    preds = (probs > 0.5).astype(int)
    acc = accuracy_score(y, preds)
    auc = roc_auc_score(y, probs)
    
    genos = ['AD' if l == 1 else 'WT' for l in y]
    print('  --- {} ---'.format(title))
    print('  Accuracy: {:.1%} ({}/{})  AUC: {:.3f}'.format(acc, sum(y == preds), len(y), auc))
    for i in range(len(y)):
        mark = 'Y' if y[i] == preds[i] else 'X'
        print('    {} ({}) prob={:.3f} [{}]'.format(animals[i], genos[i], probs[i], mark))
    
    return {'acc': acc, 'auc': auc, 'labels': y, 'probs': probs, 'preds': preds,
            'animals': list(animals), 'genos': genos, 'title': title}


# Extract features
graph_feat_df = extract_graph_features(graphs)
feat_cols = [c for c in graph_feat_df.columns if c not in ['label', 'animal', 'age_group']]
print('Extracted {} graph-level features'.format(len(feat_cols)))
print('Features:', feat_cols)

# Run baseline on all 3 splits
baseline_results = {}

print('\n' + '=' * 70)
print('BASELINE (LogReg) -- ALL')
print('=' * 70)
baseline_results['all'] = run_logreg_loao(graph_feat_df, feat_cols, 'Baseline (All)')

for age_grp in ['OLD', 'YOUNG']:
    print('\n' + '=' * 70)
    print('BASELINE (LogReg) -- {}'.format(age_grp))
    print('=' * 70)
    subset = graph_feat_df[graph_feat_df['age_group'] == age_grp].copy().reset_index(drop=True)
    n_ad = sum(subset['label'] == 1)
    n_wt = len(subset) - n_ad
    print('  {} animals: {} WT, {} AD'.format(len(subset), n_wt, n_ad))
    if n_ad < 2 or n_wt < 2:
        print('  SKIPPED')
        continue
    baseline_results[age_grp.lower()] = run_logreg_loao(
        subset, feat_cols, 'Baseline ({})'.format(age_grp))

---
## Head-to-Head: GNN vs Baseline

In [ ]:
# =============================================================================
# COMPARISON TABLE & PLOTS
# =============================================================================
print('=' * 70)
print('COMPARISON: GNN vs Baseline (Graph Features + LogReg)')
print('=' * 70)
print()
print('{:<25} {:<10} {:<12} {:<12}'.format('Approach', 'Split', 'Accuracy', 'AUC'))
print('-' * 60)

for split in ['all', 'old', 'young']:
    if split in gnn_results:
        r = gnn_results[split]
        print('{:<25} {:<10} {:<12.1%} {:<12.3f}'.format('GNN (GCN)', split.upper(), r['acc'], r['auc']))
    if split in baseline_results:
        r = baseline_results[split]
        print('{:<25} {:<10} {:<12.1%} {:<12.3f}'.format('Baseline (LogReg)', split.upper(), r['acc'], r['auc']))
    print()

# ROC comparison
splits = [k for k in ['all', 'old', 'young'] if k in gnn_results]
fig, axes = plt.subplots(1, len(splits), figsize=(6 * len(splits), 5))
if len(splits) == 1:
    axes = [axes]
split_labels = {'all': 'All Animals', 'old': 'OLD (>212d)', 'young': 'YOUNG (<=212d)'}

for i, split in enumerate(splits):
    ax = axes[i]
    
    if split in gnn_results:
        r = gnn_results[split]
        fpr, tpr, _ = roc_curve(r['labels'], r['probs'])
        ax.plot(fpr, tpr, 'b-', lw=2, label='GNN ({:.3f})'.format(r['auc']))
    
    if split in baseline_results:
        r = baseline_results[split]
        fpr, tpr, _ = roc_curve(r['labels'], r['probs'])
        ax.plot(fpr, tpr, 'r--', lw=2, label='Baseline ({:.3f})'.format(r['auc']))
    
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.set_title(split_labels[split], fontweight='bold')
    ax.legend(fontsize=11)

plt.suptitle('GNN vs Baseline: ROC Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Graph Embeddings Visualization

In [ ]:
# =============================================================================
# VISUALIZE GRAPH EMBEDDINGS (from a full model trained on all data)
# =============================================================================
from sklearn.decomposition import PCA

print('Training full model for embedding visualization...')
in_channels = graphs[0].x.shape[1]

# Train on all data (just for visualization, not for evaluation)
torch.manual_seed(42)
model_vis = GNNClassifier(in_channels=in_channels, hidden_channels=64, num_layers=3).to(DEVICE)
optimizer_vis = torch.optim.Adam(model_vis.parameters(), lr=0.001, weight_decay=1e-3)
criterion_vis = nn.CrossEntropyLoss()

all_batch = Batch.from_data_list(graphs).to(DEVICE)
model_vis.train()
for epoch in range(300):
    optimizer_vis.zero_grad()
    out = model_vis(all_batch)
    loss = criterion_vis(out, all_batch.y)
    loss.backward()
    optimizer_vis.step()

# Extract embeddings
model_vis.eval()
with torch.no_grad():
    embeddings = []
    for g in graphs:
        g_dev = g.clone().to(DEVICE)
        if not hasattr(g_dev, 'batch') or g_dev.batch is None:
            g_dev.batch = torch.zeros(g_dev.x.shape[0], dtype=torch.long, device=DEVICE)
        emb = model_vis.get_embedding(g_dev)
        embeddings.append(emb.cpu().numpy().flatten())

embeddings = np.array(embeddings)
labels = np.array([g.y.item() for g in graphs])
animals = [g.animal for g in graphs]
age_groups = [g.age_group for g in graphs]

# PCA to 2D
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(embeddings)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Color by genotype
ax = axes[0]
for label, color, name in [(0, 'steelblue', 'WT'), (1, 'crimson', 'AD')]:
    mask = labels == label
    ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], c=color, label=name,
              s=100, edgecolors='black', zorder=5)
for i, name in enumerate(animals):
    ax.annotate(name, (emb_2d[i, 0], emb_2d[i, 1]), fontsize=7,
               ha='center', va='bottom', textcoords='offset points', xytext=(0, 5))
ax.set_xlabel('PC1 ({:.1%} var)'.format(pca.explained_variance_ratio_[0]))
ax.set_ylabel('PC2 ({:.1%} var)'.format(pca.explained_variance_ratio_[1]))
ax.set_title('GNN Embeddings: by Genotype', fontweight='bold')
ax.legend()

# Color by age group
ax = axes[1]
group_colors = {'WT_OLD': 'darkblue', 'WT_YOUNG': 'lightblue',
                'AD_OLD': 'darkred', 'AD_YOUNG': 'lightcoral'}
for i in range(len(graphs)):
    geno = 'AD' if labels[i] == 1 else 'WT'
    group = '{}_{}'.format(geno, age_groups[i])
    ax.scatter(emb_2d[i, 0], emb_2d[i, 1], c=group_colors[group],
              s=100, edgecolors='black', zorder=5, label=group)
    ax.annotate(animals[i], (emb_2d[i, 0], emb_2d[i, 1]), fontsize=7,
               ha='center', va='bottom', textcoords='offset points', xytext=(0, 5))
# Deduplicate legend
handles, labs = ax.get_legend_handles_labels()
by_label = dict(zip(labs, handles))
ax.legend(by_label.values(), by_label.keys())
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('GNN Embeddings: by Group', fontweight='bold')

plt.suptitle('Learned Graph Representations (PCA of GNN embeddings)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Summary

In [ ]:
# =============================================================================
# FINAL SUMMARY
# =============================================================================
print('=' * 70)
print('FINAL SUMMARY')
print('=' * 70)
print()
print('{:<25} {:<10} {:<12} {:<12}'.format('Approach', 'Split', 'Accuracy', 'AUC'))
print('-' * 60)
for split in ['all', 'old', 'young']:
    if split in gnn_results:
        r = gnn_results[split]
        print('{:<25} {:<10} {:<12.1%} {:<12.3f}'.format('GNN (GCN)', split.upper(), r['acc'], r['auc']))
    if split in baseline_results:
        r = baseline_results[split]
        print('{:<25} {:<10} {:<12.1%} {:<12.3f}'.format('Baseline (LogReg)', split.upper(), r['acc'], r['auc']))
    print()

print('\nKEY INSIGHTS:')
print('-' * 60)

if 'all' in gnn_results and 'all' in baseline_results:
    gnn_auc = gnn_results['all']['auc']
    bl_auc = baseline_results['all']['auc']
    if gnn_auc > bl_auc:
        print('1. GNN ({:.3f}) > Baseline ({:.3f})'.format(gnn_auc, bl_auc))
        print('   -> Graph structure (message passing) adds value beyond summary statistics')
    else:
        print('1. Baseline ({:.3f}) >= GNN ({:.3f})'.format(bl_auc, gnn_auc))
        print('   -> Summary statistics capture most discriminative information')
        print('   -> With only 20 graphs, GNN may not have enough data to learn well')

if 'old' in gnn_results and 'young' in gnn_results:
    old_auc = gnn_results['old']['auc']
    young_auc = gnn_results['young']['auc']
    print('2. OLD AUC={:.3f} vs YOUNG AUC={:.3f} (diff={:+.3f})'.format(
        old_auc, young_auc, old_auc - young_auc))
    if old_auc > young_auc:
        print('   -> Connectivity differences are more pronounced in symptomatic mice')
    else:
        print('   -> Connectivity differences detectable even pre-symptomatically')

print()
print('METHODOLOGY:')
print('- Leave-One-Animal-Out CV (animal-level classification)')
print('- GNN: 3-layer GCN, 64 hidden dim, mean+max pool, 5 runs averaged')
print('- Baseline: 18 hand-crafted graph features + L2-regularized LogReg')
print('- Graphs: GLM connectivity (REM sleep), 4 brain regions combined')
print('- Node features: cell type, firing rate, degree, region')
print('- Edge features: coupling weight (directed, signed)')
print('- Age cutoff: {} days (APP/PS1 symptom onset)'.format(AGE_CUTOFF))